# Food-Domain Aspect-Based Sentiment Analysis

SageMaker Studio notebook for the project's main research contribution: identifying **what** customers discuss and the sentiment attached to each food-related aspect.

The notebook implements a defensible weakly supervised baseline:

1. LDA topic discovery to inspect themes and find missing vocabulary.
2. Literature/domain-seeded aspect lexicons to create weak multi-label targets.
3. TF-IDF + One-vs-Rest Logistic Regression to generalize beyond exact keywords.
4. VADER as an explicitly labelled sentence-sentiment baseline.
5. A final inference cell for testing new review text.

> Weak-label agreement is not the same as human-labelled accuracy. Do not present the weak test metrics as final model performance.

## 1. Install dependencies

In [ ]:
import importlib
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pandas", "pyarrow", "scikit-learn", "joblib",
    "matplotlib", "seaborn", "boto3", "vaderSentiment",
])
importlib.invalidate_caches()
print(f"Installed dependencies into: {sys.executable}")

## 2. Imports and configuration

In [ ]:
import importlib
import json
import platform
import re
import subprocess
import sys
import tarfile
import time
from datetime import datetime, timezone
from pathlib import Path

import boto3
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns
import sklearn
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, hamming_loss
from sklearn.multiclass import OneVsRestClassifier
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
except ModuleNotFoundError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "vaderSentiment"
    ])
    importlib.invalidate_caches()
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

SEED = 42
BUCKET = "amazon-food-reviews-ml-model"
ASPECT_INPUT_PREFIX = "gold/aspect_sentences/"
MODEL_OUTPUT_PREFIX = "models/aspect-baseline/"
MAX_MODEL_SENTENCES = 200_000
MAX_TOPIC_SENTENCES = 60_000
MAX_TFIDF_FEATURES = 80_000
ASPECT_THRESHOLD = 0.50

np.random.seed(SEED)
run_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
session = boto3.Session()
region = session.region_name or "eu-north-1"
s3 = session.client("s3", region_name=region)

print(f"Region: {region}")
print(f"Aspect input: s3://{BUCKET}/{ASPECT_INPUT_PREFIX}")

## 3. Seed the food-domain aspect taxonomy

These seeds provide weak supervision. Topic discovery and manual review should refine them; they are not assumed to be complete ground truth.

In [ ]:
ASPECT_LEXICON = {
    "taste": [
        "taste", "tastes", "tasty", "flavor", "flavour", "flavored",
        "delicious", "yummy", "bland", "bitter", "sweet", "sour",
        "salty", "savory", "spicy", "aftertaste",
    ],
    "freshness": [
        "fresh", "freshness", "stale", "spoiled", "rotten", "mold",
        "moldy", "expired", "expiry", "expiration", "shelf life",
    ],
    "texture": [
        "texture", "crunchy", "crispy", "crisp", "chewy", "soft",
        "hard", "soggy", "dry", "moist", "smooth", "creamy",
    ],
    "packaging": [
        "package", "packaging", "packed", "packet", "box", "bag",
        "bottle", "container", "seal", "sealed", "lid", "wrapper",
        "leak", "leaked", "crushed", "damaged", "broken",
    ],
    "delivery": [
        "delivery", "shipping", "shipped", "arrived", "arrival",
        "delivered", "late", "delayed", "courier", "melted", "prime",
    ],
    "price_value": [
        "price", "priced", "expensive", "cheap", "cost", "costly",
        "value", "worth", "overpriced", "affordable", "money",
        "deal", "bargain",
    ],
    "ingredients_health": [
        "ingredient", "ingredients", "healthy", "health", "organic",
        "natural", "sugar", "sugar free", "gluten", "gluten free",
        "calorie", "calories", "diet", "protein", "vegan",
        "allergy", "allergic", "preservative", "additive",
    ],
    "portion_quantity": [
        "portion", "size", "quantity", "amount", "serving", "count",
        "small", "large", "tiny", "generous", "ounces", "weight",
    ],
    "quality": [
        "quality", "premium", "authentic", "genuine", "inferior",
        "poor quality", "high quality", "low quality",
    ],
}
ASPECTS = list(ASPECT_LEXICON)
ASPECT_PATTERNS = {
    aspect: re.compile(
        r"\b(?:" + "|".join(re.escape(term) for term in terms) + r")\b",
        flags=re.IGNORECASE,
    )
    for aspect, terms in ASPECT_LEXICON.items()
}

print(f"Configured {len(ASPECTS)} aspects: {', '.join(ASPECTS)}")

## 4. Download Gold aspect-sentence Parquet files

In [ ]:
local_aspect_dir = Path("data/aspect_sentences") / run_id
local_aspect_dir.mkdir(parents=True, exist_ok=True)

parquet_keys = []
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=BUCKET, Prefix=ASPECT_INPUT_PREFIX):
    for item in page.get("Contents", []):
        if item["Key"].endswith(".parquet"):
            parquet_keys.append(item["Key"])

if not parquet_keys:
    raise FileNotFoundError(
        f"No aspect Parquet files at s3://{BUCKET}/{ASPECT_INPUT_PREFIX}"
    )

for key in parquet_keys:
    destination = local_aspect_dir / Path(key[len(ASPECT_INPUT_PREFIX):])
    destination.parent.mkdir(parents=True, exist_ok=True)
    s3.download_file(BUCKET, key, str(destination))

local_parquet_files = sorted(local_aspect_dir.rglob("*.parquet"))
total_sentence_rows = sum(pq.ParquetFile(path).metadata.num_rows for path in local_parquet_files)
print(f"Downloaded {len(local_parquet_files)} files containing {total_sentence_rows:,} sentences")

## 5. Create a deterministic development sample

The complete sentence layer may contain millions of rows. This cell streams Parquet batches and samples by `sentence_id` hash without loading the full dataset into memory.

In [ ]:
MODEL_COLUMNS = [
    "sentence_id", "record_id", "product_id", "score",
    "sentiment_class", "review_timestamp", "review_year",
    "helpfulness_numerator", "helpfulness_denominator",
    "helpfulness_ratio", "sentence_text", "sentence_normalized",
]
sample_probability = min(1.0, MAX_MODEL_SENTENCES / total_sentence_rows)
hash_limit = int(sample_probability * np.iinfo(np.uint64).max)
sampled_batches = []

for parquet_path in local_parquet_files:
    parquet_file = pq.ParquetFile(parquet_path)
    missing = sorted(set(MODEL_COLUMNS) - set(parquet_file.schema.names))
    if missing:
        raise ValueError(f"{parquet_path} is missing columns: {missing}")
    for batch in parquet_file.iter_batches(batch_size=50_000, columns=MODEL_COLUMNS):
        frame = batch.to_pandas()
        hashes = pd.util.hash_pandas_object(
            frame["sentence_id"].astype(str), index=False
        ).to_numpy(dtype=np.uint64)
        selected = frame.loc[hashes <= hash_limit]
        if not selected.empty:
            sampled_batches.append(selected)

if not sampled_batches:
    raise ValueError("No sentences were selected; increase MAX_MODEL_SENTENCES")
model_data = pd.concat(sampled_batches, ignore_index=True)
if len(model_data) > MAX_MODEL_SENTENCES:
    model_data = model_data.sample(MAX_MODEL_SENTENCES, random_state=SEED)
model_data = model_data.dropna(subset=["sentence_id", "record_id", "sentence_text"]).copy()
model_data["sentence_text"] = model_data["sentence_text"].astype(str).str.strip()
model_data = model_data[model_data["sentence_text"].ne("")].drop_duplicates("sentence_id")
model_data = model_data.reset_index(drop=True)

if model_data.empty:
    raise ValueError("Deterministic aspect-model sample is empty")
print(f"Development sample: {len(model_data):,} sentences")
display(model_data.head())

## 6. Exploratory LDA topic discovery

LDA helps inspect whether food-domain themes and unanticipated vocabulary occur in the corpus. Topic/seed overlap is descriptive evidence—not supervised validation.

In [ ]:
topic_data = model_data.sample(
    n=min(MAX_TOPIC_SENTENCES, len(model_data)), random_state=SEED
)
count_vectorizer = CountVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=10,
    max_df=0.70,
    max_features=6_000,
    ngram_range=(1, 2),
)
topic_counts = count_vectorizer.fit_transform(topic_data["sentence_text"])
lda = LatentDirichletAllocation(
    n_components=12,
    learning_method="online",
    max_iter=15,
    random_state=SEED,
    n_jobs=-1,
)
lda.fit(topic_counts)
vocabulary = np.asarray(count_vectorizer.get_feature_names_out())
seed_sets = {aspect: {term.lower() for term in terms} for aspect, terms in ASPECT_LEXICON.items()}

topic_rows = []
for topic_index, weights in enumerate(lda.components_):
    top_terms = vocabulary[np.argsort(weights)[-15:][::-1]].tolist()
    overlaps = {
        aspect: len(set(top_terms) & seeds) for aspect, seeds in seed_sets.items()
    }
    best_aspect = max(overlaps, key=overlaps.get)
    topic_rows.append(
        {
            "topic": topic_index,
            "top_terms": ", ".join(top_terms),
            "suggested_aspect": best_aspect if overlaps[best_aspect] else "emergent_or_unclear",
            "seed_overlap": overlaps[best_aspect],
        }
    )
topic_summary = pd.DataFrame(topic_rows)
display(topic_summary)

## 7. Create weak multi-label aspect targets

In [ ]:
def weak_aspect_labels(text):
    return [int(bool(ASPECT_PATTERNS[aspect].search(text))) for aspect in ASPECTS]


weak_label_matrix = np.asarray(
    [weak_aspect_labels(text) for text in model_data["sentence_text"]],
    dtype=np.int8,
)
for index, aspect in enumerate(ASPECTS):
    model_data[f"weak_{aspect}"] = weak_label_matrix[:, index]

weak_counts = pd.Series(weak_label_matrix.sum(axis=0), index=ASPECTS, name="weak_positive_sentences")
display(weak_counts.sort_values(ascending=False).to_frame())
print(f"Sentences with at least one seeded aspect: {(weak_label_matrix.sum(axis=1) > 0).mean():.1%}")

if (weak_counts < 30).any():
    raise ValueError(
        "Some aspects have fewer than 30 weak positives. Expand their seeds or sample more data: "
        + str(weak_counts[weak_counts < 30].to_dict())
    )

## 8. Make review-grouped development splits

All sentences from one review receive the same split, preventing sentence leakage between training and evaluation.

In [ ]:
review_hash = pd.util.hash_pandas_object(
    model_data["record_id"].astype(str), index=False
).to_numpy(dtype=np.uint64) % 100
model_data["development_split"] = np.where(
    review_hash < 80, "train", np.where(review_hash < 90, "validation", "test")
)

split_indices = {
    split: np.flatnonzero(model_data["development_split"].to_numpy() == split)
    for split in ["train", "validation", "test"]
}
for split, indices in split_indices.items():
    if len(indices) == 0:
        raise ValueError(f"{split} split is empty")

review_sets = {
    split: set(model_data.iloc[indices]["record_id"])
    for split, indices in split_indices.items()
}
if review_sets["train"] & review_sets["validation"] or review_sets["train"] & review_sets["test"] or review_sets["validation"] & review_sets["test"]:
    raise ValueError("Review IDs overlap across aspect-model splits")

display(pd.crosstab(model_data["development_split"], columns="sentences"))

## 9. Train the multi-label aspect detector

TF-IDF is fitted only on training sentences. Class-balanced one-vs-rest classifiers learn broader lexical contexts around the seed matches.

In [ ]:
train_indices = split_indices["train"]
validation_indices = split_indices["validation"]
test_indices = split_indices["test"]

vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.98,
    max_features=MAX_TFIDF_FEATURES,
    sublinear_tf=True,
    dtype=np.float32,
)
X_train = vectorizer.fit_transform(model_data.iloc[train_indices]["sentence_text"]).tocsr()
X_validation = vectorizer.transform(model_data.iloc[validation_indices]["sentence_text"]).tocsr()
X_test = vectorizer.transform(model_data.iloc[test_indices]["sentence_text"]).tocsr()
# Canonical CSR matrices prevent SciPy from needing an in-place sort during fit.
for matrix in (X_train, X_validation, X_test):
    matrix.sum_duplicates()
    matrix.sort_indices()
Y_train = weak_label_matrix[train_indices]
Y_validation = weak_label_matrix[validation_indices]
Y_test = weak_label_matrix[test_indices]
train_positive_counts = Y_train.sum(axis=0)
train_negative_counts = len(Y_train) - train_positive_counts
invalid_training_aspects = [
    ASPECTS[index]
    for index in range(len(ASPECTS))
    if train_positive_counts[index] == 0 or train_negative_counts[index] == 0
]
if invalid_training_aspects:
    raise ValueError(
        f"Aspect classifiers need positive and negative training examples: {invalid_training_aspects}"
    )

aspect_model = OneVsRestClassifier(
    LogisticRegression(
        C=1.0,
        class_weight="balanced",
        solver="liblinear",
        max_iter=500,
        random_state=SEED,
    ),
    # SageMaker/loky can expose sparse matrices as read-only in child processes.
    # Sequential fitting is reliable here and only trains one classifier per aspect.
    n_jobs=1,
)
started = time.perf_counter()
aspect_model.fit(X_train, Y_train)
print(f"Trained {len(ASPECTS)} aspect classifiers in {time.perf_counter() - started:.1f}s")
print(f"TF-IDF vocabulary: {len(vectorizer.vocabulary_):,}")

## 10. Measure held-out weak-label agreement

These values measure how well the classifier generalizes the seed rules. They must later be replaced or supplemented with metrics from the human annotation file.

In [ ]:
validation_probabilities = aspect_model.predict_proba(X_validation)
test_probabilities = aspect_model.predict_proba(X_test)
validation_predictions = (validation_probabilities >= ASPECT_THRESHOLD).astype(np.int8)
test_predictions = (test_probabilities >= ASPECT_THRESHOLD).astype(np.int8)

weak_metrics = {
    "validation_micro_f1": float(f1_score(Y_validation, validation_predictions, average="micro", zero_division=0)),
    "validation_macro_f1": float(f1_score(Y_validation, validation_predictions, average="macro", zero_division=0)),
    "test_micro_f1": float(f1_score(Y_test, test_predictions, average="micro", zero_division=0)),
    "test_macro_f1": float(f1_score(Y_test, test_predictions, average="macro", zero_division=0)),
    "test_hamming_loss": float(hamming_loss(Y_test, test_predictions)),
}
display(pd.DataFrame([weak_metrics]))

per_aspect_report = classification_report(
    Y_test,
    test_predictions,
    target_names=ASPECTS,
    output_dict=True,
    zero_division=0,
)
per_aspect_df = pd.DataFrame(per_aspect_report).T.loc[ASPECTS, ["precision", "recall", "f1-score", "support"]]
display(per_aspect_df.sort_values("f1-score"))

## 11. Apply the detector and VADER sentiment baseline

VADER supplies a generic sentence-polarity baseline. It is not yet a validated food-domain aspect-sentiment model. Because Gold contains one aspect-relevant sentence per row, its score is attached to every detected aspect in that sentence.

In [ ]:
X_all = vectorizer.transform(model_data["sentence_text"])
all_probabilities = aspect_model.predict_proba(X_all)
all_predictions = (all_probabilities >= ASPECT_THRESHOLD).astype(np.int8)
model_data["seed_aspects"] = [
    [ASPECTS[j] for j, value in enumerate(row) if value] for row in weak_label_matrix
]
model_data["predicted_aspects"] = [
    [ASPECTS[j] for j, value in enumerate(row) if value] for row in all_predictions
]

sentiment_analyzer = SentimentIntensityAnalyzer()
model_data["vader_compound"] = model_data["sentence_text"].map(
    lambda text: sentiment_analyzer.polarity_scores(text)["compound"]
)
model_data["vader_sentiment"] = np.select(
    [model_data["vader_compound"] >= 0.05, model_data["vader_compound"] <= -0.05],
    ["positive", "negative"],
    default="neutral",
)

aspect_predictions = (
    model_data[model_data["predicted_aspects"].map(bool)]
    .explode("predicted_aspects")
    .rename(columns={"predicted_aspects": "aspect"})
)
aspect_predictions["aspect_probability"] = [
    all_probabilities[index, ASPECTS.index(aspect)]
    for index, aspect in zip(aspect_predictions.index, aspect_predictions["aspect"])
]
aspect_predictions = aspect_predictions.reset_index(drop=True)

print(f"Sentence-aspect predictions: {len(aspect_predictions):,}")
display(aspect_predictions[["sentence_text", "aspect", "aspect_probability", "vader_sentiment", "vader_compound"]].head(20))

In [ ]:
aspect_summary = (
    aspect_predictions.groupby("aspect")
    .agg(
        mentions=("sentence_id", "count"),
        mean_aspect_probability=("aspect_probability", "mean"),
        mean_vader_compound=("vader_compound", "mean"),
    )
    .sort_values("mentions", ascending=False)
)
sentiment_mix = pd.crosstab(
    aspect_predictions["aspect"],
    aspect_predictions["vader_sentiment"],
    normalize="index",
)
display(aspect_summary)
display(sentiment_mix)

## 12. Save predictions, topics, and model artifacts

In [ ]:
artifact_dir = Path("artifacts") / run_id / "aspect_model"
artifact_dir.mkdir(parents=True, exist_ok=True)
model_file = artifact_dir / "aspect_model.joblib"
metadata_file = artifact_dir / "metadata.json"
topics_file = artifact_dir / "lda_topics.csv"
predictions_file = artifact_dir / "aspect_predictions_sample.parquet"
archive_file = artifact_dir / "aspect_model.tar.gz"

joblib.dump(
    {
        "vectorizer": vectorizer,
        "aspect_model": aspect_model,
        "aspects": ASPECTS,
        "aspect_lexicon": ASPECT_LEXICON,
        "threshold": ASPECT_THRESHOLD,
    },
    model_file,
)
topic_summary.to_csv(topics_file, index=False)
prediction_columns = [
    "sentence_id", "record_id", "product_id", "score",
    "review_timestamp", "review_year", "helpfulness_numerator",
    "helpfulness_denominator", "helpfulness_ratio", "sentence_text",
    "aspect", "aspect_probability", "vader_sentiment", "vader_compound",
]
aspect_predictions[prediction_columns].to_parquet(predictions_file, index=False)

metadata = {
    "run_id": run_id,
    "method": "seeded weak supervision + TF-IDF One-vs-Rest Logistic Regression",
    "research_status": "weak baseline; human evaluation required",
    "input": f"s3://{BUCKET}/{ASPECT_INPUT_PREFIX}",
    "sampled_sentences": int(len(model_data)),
    "aspects": ASPECTS,
    "threshold": ASPECT_THRESHOLD,
    "weak_label_metrics": weak_metrics,
    "per_aspect_weak_report": json.loads(per_aspect_df.to_json(orient="index")),
    "versions": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
    },
}
with metadata_file.open("w", encoding="utf-8") as handle:
    json.dump(metadata, handle, indent=2)

with tarfile.open(archive_file, "w:gz") as archive:
    archive.add(model_file, arcname="aspect_model.joblib")
    archive.add(metadata_file, arcname="metadata.json")
    archive.add(topics_file, arcname="lda_topics.csv")

output_base = f"{MODEL_OUTPUT_PREFIX}{run_id}/"
for local_file in [archive_file, metadata_file, topics_file, predictions_file]:
    s3.upload_file(str(local_file), BUCKET, output_base + local_file.name)

print(f"Aspect model artifacts: s3://{BUCKET}/{output_base}")

## 13. Test the model with a new review

Change `review_input` to any food review and run this final cell. The output shows detected aspects, model confidence, and sentence-level sentiment.

In [ ]:
def predict_review_aspects(review_text):
    if not isinstance(review_text, str) or not review_text.strip():
        raise ValueError("Enter a non-empty review string.")

    sentences = [
        sentence.strip()
        for sentence in re.split(r"(?<=[.!?])\s+|[\r\n]+", review_text.strip())
        if sentence.strip()
    ]
    features = vectorizer.transform(sentences)
    probabilities = aspect_model.predict_proba(features)
    rows = []

    for sentence_index, (sentence, scores) in enumerate(zip(sentences, probabilities), start=1):
        sentiment_scores = sentiment_analyzer.polarity_scores(sentence)
        compound = sentiment_scores["compound"]
        sentiment = "positive" if compound >= 0.05 else "negative" if compound <= -0.05 else "neutral"
        detected = np.flatnonzero(scores >= ASPECT_THRESHOLD)

        for aspect_index in detected:
            rows.append({
                "sentence_number": sentence_index,
                "sentence": sentence,
                "aspect": ASPECTS[aspect_index],
                "aspect_confidence": round(float(scores[aspect_index]), 4),
                "sentiment": sentiment,
                "sentiment_score": compound,
            })

        if len(detected) == 0:
            best_index = int(np.argmax(scores))
            rows.append({
                "sentence_number": sentence_index,
                "sentence": sentence,
                "aspect": "no aspect above threshold",
                "aspect_confidence": round(float(scores[best_index]), 4),
                "sentiment": sentiment,
                "sentiment_score": compound,
            })

    return pd.DataFrame(rows)


# Replace this text with any review you want to test.
review_input = "The chocolate tastes delicious. The packaging was damaged. Delivery was very late."
prediction_output = predict_review_aspects(review_input)
display(prediction_output)